# Notebook 08 — Classical ML Baselines

This notebook evaluates classical machine learning baselines for the water-quality sequence-to-class forecasting task.

The goal is to test whether tabular classical models can compete with the deep-learning baselines trained in the previous notebooks. The same preprocessed tensors from Notebook 02 are reused here to ensure a fair comparison. Each input sample originally has shape `(12, 8)`, representing 12 monthly time steps and 8 water-quality features. Since classical machine learning models expect tabular input, each sequence is flattened into a vector of 96 features.

The notebook evaluates both local and centralized classical ML settings:

- **Local setting:** one model is trained separately for each client/country.
- **Centralized setting:** one model is trained on pooled training data from England, Ireland, and USA, then evaluated separately on each client test set.

The main evaluation metric is macro-F1 over present classes, while accuracy, balanced accuracy, all-class macro-F1, weighted-F1, per-class reports, and confusion matrices are also reported.

The final cross-family comparison with GRU, LSTM, Transformer, and federated models is left to Notebook 09. This notebook only produces the classical ML results needed for that comparison.


## Phase 1 — Setup, tensor loading, and tabular conversion

This phase prepares the notebook environment, loads the preprocessed tensors created in Notebook 02, verifies their shapes, and converts sequence tensors from `(N, 12, 8)` to `(N, 96)` for classical machine learning models.

## Section 0 — Setup

In [1]:
# 0.1 — Imports

import os
import json
import time
import warnings
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    precision_recall_fscore_support,
    f1_score,
    classification_report,
    confusion_matrix,
)

from sklearn.utils.class_weight import compute_sample_weight

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

SEED = 42
np.random.seed(SEED)

print(f"numpy  : {np.__version__}")
print(f"pandas : {pd.__version__}")

numpy  : 2.4.6
pandas : 2.3.3


In [2]:
# 0.2 — Project constants

ACTIVE_CLIENTS = ["England", "Ireland", "USA"]
SPLITS = ["train", "val", "test"]

WINDOW = 12
N_FEATURES = 8
N_CLASSES = 5

FEATURE_COLS = ["NH3", "BOD", "DO", "PO4", "pH", "Temp", "Nitrogen", "Nitrate"]

CLASS_TO_INT = {
    "Excellent": 0,
    "Good": 1,
    "Fair": 2,
    "Marginal": 3,
    "Poor": 4,
}

INT_TO_CLASS = {v: k for k, v in CLASS_TO_INT.items()}
CLASS_NAMES = [INT_TO_CLASS[i] for i in range(N_CLASSES)]

assert len(FEATURE_COLS) == N_FEATURES
assert len(CLASS_NAMES) == N_CLASSES

print("Configuration locked.")
print(f"Active clients : {ACTIVE_CLIENTS}")
print(f"Window         : {WINDOW} months")
print(f"Features       : {FEATURE_COLS}")
print(f"Classes        : {CLASS_NAMES}")

Configuration locked.
Active clients : ['England', 'Ireland', 'USA']
Window         : 12 months
Features       : ['NH3', 'BOD', 'DO', 'PO4', 'pH', 'Temp', 'Nitrogen', 'Nitrate']
Classes        : ['Excellent', 'Good', 'Fair', 'Marginal', 'Poor']


In [3]:
# 0.3 — Auto-discover paths

# This notebook should not depend on a fixed Kaggle dataset slug.
# It searches /kaggle/input for the preprocessing manifest produced by N02,
# then derives the tensor directory from that discovered location.

def find_dir(marker, start="/kaggle/input"):
    """Return the first directory under start that contains marker."""
    for root, dirs, files in os.walk(start):
        if marker in files:
            return Path(root)
    return None

N02_DIR = find_dir("preprocess_manifest.json")
assert N02_DIR is not None, "N02 preprocessing outputs not found: preprocess_manifest.json is missing under /kaggle/input."

INPUT_DIR = N02_DIR
TENSOR_DIR = INPUT_DIR / "tensors"
MANIFEST_PATH = INPUT_DIR / "preprocess_manifest.json"

OUTPUT_DIR = Path("/kaggle/working/classical_ml_baselines")

# Start from a clean output folder so old files from previous runs cannot be mixed
# with the current N08 results.
if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

METRIC_DIR = OUTPUT_DIR / "metrics"
PRED_DIR = OUTPUT_DIR / "predictions"
FIG_DIR = OUTPUT_DIR / "figures"
MODEL_DIR = OUTPUT_DIR / "models"
AUDIT_DIR = OUTPUT_DIR / "audit"

for d in [METRIC_DIR, PRED_DIR, FIG_DIR, MODEL_DIR, AUDIT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

assert TENSOR_DIR.exists(), f"Tensor directory not found: {TENSOR_DIR}"
assert MANIFEST_PATH.exists(), f"Manifest file not found: {MANIFEST_PATH}"

print(f"N02 directory     : {INPUT_DIR}")
print(f"Tensor directory  : {TENSOR_DIR}")
print(f"Manifest path     : {MANIFEST_PATH}")
print(f"Output directory  : {OUTPUT_DIR}")


N02 directory     : /kaggle/input/datasets/hebacheblal/preprocess-outputs
Tensor directory  : /kaggle/input/datasets/hebacheblal/preprocess-outputs/tensors
Manifest path     : /kaggle/input/datasets/hebacheblal/preprocess-outputs/preprocess_manifest.json
Output directory  : /kaggle/working/classical_ml_baselines


In [4]:
# 0.4 — Plot defaults and saving helpers

plt.rcParams.update({
    "figure.figsize":    (10, 5),
    "figure.dpi":        110,
    "savefig.dpi":       250,
    "savefig.bbox":      "tight",
    "font.size":         11,
    "axes.titlesize":    12,
    "axes.labelsize":    11,
    "axes.grid":         True,
    "grid.alpha":        0.25,
    "axes.spines.top":   False,
    "axes.spines.right": False,
})


def save_table(df, name, directory=METRIC_DIR, verbose=True):
    """Save a DataFrame as CSV and print only the file name."""
    path = directory / f"{name}.csv"
    df.to_csv(path, index=False)

    if verbose:
        print(f"✓ saved table: {path.name}")

    return path


def save_figure(name, fig=None, formats=("png", "pdf"), verbose=True):
    """Save a figure and print only the figure name."""
    fig = fig or plt.gcf()

    saved_names = []
    for ext in formats:
        path = FIG_DIR / f"{name}.{ext}"
        fig.savefig(path)
        saved_names.append(path.name)

    if verbose:
        print(f"✓ saved figure: {', '.join(saved_names)}")

## Section 1 — Load preprocessed tensors from Notebook 02

The same tensors used in the deep-learning notebooks are loaded here. This keeps the comparison fair because classical ML models receive exactly the same input information, only reshaped from sequence format `(N, 12, 8)` into tabular format `(N, 96)`.


In [5]:
# 1.1 — Load tensors

data = {}

print(f"Loading tensors from: {TENSOR_DIR}\n")

for client in ACTIVE_CLIENTS:
    npz_path = TENSOR_DIR / f"{client.lower()}_stage1_sequences.npz"
    assert npz_path.exists(), f"Missing tensor file: {npz_path}"

    archive = np.load(npz_path)

    data[client] = {}

    for split in SPLITS:
        data[client][split] = {
            "X": archive[f"X_{split}"],
            "y": archive[f"y_{split}"],
            "mask": archive[f"mask_{split}"],
            "site_ids": archive[f"site_ids_{split}"],
            "target_dates": archive[f"target_dates_{split}"],
        }

    archive.close()

    split_sizes = " | ".join(
        f"{split}={data[client][split]['X'].shape[0]:,}"
        for split in SPLITS
    )
    print(f"{client:<8} {split_sizes}")

print("\n✓ All client tensors loaded.")

Loading tensors from: /kaggle/input/datasets/hebacheblal/preprocess-outputs/tensors

England  train=76,530 | val=8,490 | test=11,942
Ireland  train=27,778 | val=4,296 | test=5,887
USA      train=30,222 | val=1,770 | test=1,979

✓ All client tensors loaded.


In [6]:
# 1.2 — Verify tensor shapes and temporal ranges

shape_rows = []

for client in ACTIVE_CLIENTS:
    for split in SPLITS:
        X = data[client][split]["X"]
        y = data[client][split]["y"]
        mask = data[client][split]["mask"]
        dates = pd.to_datetime(data[client][split]["target_dates"])

        assert X.ndim == 3, f"{client} {split}: X must be 3D"
        assert X.shape[1:] == (WINDOW, N_FEATURES), f"{client} {split}: unexpected X shape {X.shape}"
        assert y.ndim == 1, f"{client} {split}: y must be 1D"
        assert mask.shape == X.shape, f"{client} {split}: mask shape mismatch"
        assert X.shape[0] == y.shape[0] == mask.shape[0] == len(dates), f"{client} {split}: N mismatch"
        assert np.isfinite(X).all(), f"{client} {split}: X contains non-finite values"
        assert np.isin(y, list(range(N_CLASSES))).all(), f"{client} {split}: y contains invalid labels"

        shape_rows.append({
            "Client": client,
            "Split": split,
            "N_sequences": X.shape[0],
            "X_shape": str(X.shape),
            "y_shape": str(y.shape),
            "mask_shape": str(mask.shape),

            # Correct temporal audit: real earliest and latest target dates
            "Target_min_date": dates.min().date() if len(dates) > 0 else None,
            "Target_max_date": dates.max().date() if len(dates) > 0 else None,

            # This only checks whether rows are globally sorted by date.
            # False is not a problem after train/val/test tensors are already created.
            "Rows_globally_sorted_by_date": bool(pd.Series(dates).is_monotonic_increasing),
        })

tensor_shape_df = pd.DataFrame(shape_rows)

print("Tensor shape and temporal range verification:")
display(tensor_shape_df)

save_table(tensor_shape_df, "01_tensor_shape_verification", directory=AUDIT_DIR)

Tensor shape and temporal range verification:


,Client,Split,N_sequences,X_shape,y_shape,mask_shape,Target_min_date,Target_max_date,Rows_globally_sorted_by_date
0,England,train,76530,"(76530, 12, 8)","(76530,)","(76530, 12, 8)",2001-01-31,2017-12-31,False
1,England,val,8490,"(8490, 12, 8)","(8490,)","(8490, 12, 8)",2018-01-31,2019-12-31,False
2,England,test,11942,"(11942, 12, 8)","(11942,)","(11942, 12, 8)",2020-01-31,2023-10-31,False
3,Ireland,train,27778,"(27778, 12, 8)","(27778,)","(27778, 12, 8)",2008-01-31,2017-12-31,False
4,Ireland,val,4296,"(4296, 12, 8)","(4296,)","(4296, 12, 8)",2018-01-31,2019-12-31,False
5,Ireland,test,5887,"(5887, 12, 8)","(5887,)","(5887, 12, 8)",2020-01-31,2023-09-30,False
6,USA,train,30222,"(30222, 12, 8)","(30222,)","(30222, 12, 8)",1941-01-31,2017-12-31,False
7,USA,val,1770,"(1770, 12, 8)","(1770,)","(1770, 12, 8)",2018-01-31,2019-12-31,False
8,USA,test,1979,"(1979, 12, 8)","(1979,)","(1979, 12, 8)",2020-01-31,2023-07-31,False


✓ saved table: 01_tensor_shape_verification.csv


PosixPath('/kaggle/working/classical_ml_baselines/audit/01_tensor_shape_verification.csv')

The saved tensor rows are not globally sorted by target date. This only describes the order of samples in the saved arrays. It does not mean that the 12 time steps inside each sample are unordered. In Notebook 02, sequences were constructed after sorting each site timeline by `year_month`, then extracting consecutive 12-month windows. Therefore, each input sample is expected to preserve its internal chronological order from older to more recent months.

## Section 2 — Convert sequence tensors to tabular features

Classical ML models such as Logistic Regression, Random Forest, XGBoost, and CatBoost require 2D tabular input. Therefore, each sequence of shape `(12, 8)` is flattened into a vector of 96 features.

The first 8 features correspond to the oldest month in the input window, and the last 8 features correspond to the most recent month before the prediction target.

In [7]:
# 2.1 — Build flattened feature names

flat_feature_names = []

for step in range(WINDOW):
    lag = WINDOW - step
    for feature in FEATURE_COLS:
        flat_feature_names.append(f"t_minus_{lag}_{feature}")

assert len(flat_feature_names) == WINDOW * N_FEATURES

print(f"Number of flattened features: {len(flat_feature_names)}")
print("First 12 flattened feature names:")
display(flat_feature_names[:12])

print("Last 12 flattened feature names:")
display(flat_feature_names[-12:])

Number of flattened features: 96
First 12 flattened feature names:


['t_minus_12_NH3',
 't_minus_12_BOD',
 't_minus_12_DO',
 't_minus_12_PO4',
 't_minus_12_pH',
 't_minus_12_Temp',
 't_minus_12_Nitrogen',
 't_minus_12_Nitrate',
 't_minus_11_NH3',
 't_minus_11_BOD',
 't_minus_11_DO',
 't_minus_11_PO4']

Last 12 flattened feature names:


['t_minus_2_pH',
 't_minus_2_Temp',
 't_minus_2_Nitrogen',
 't_minus_2_Nitrate',
 't_minus_1_NH3',
 't_minus_1_BOD',
 't_minus_1_DO',
 't_minus_1_PO4',
 't_minus_1_pH',
 't_minus_1_Temp',
 't_minus_1_Nitrogen',
 't_minus_1_Nitrate']

In [8]:
# 2.2 — Convert all X tensors from (N, 12, 8) to (N, 96)

tabular_data = {}

for client in ACTIVE_CLIENTS:
    tabular_data[client] = {}

    for split in SPLITS:
        X_seq = data[client][split]["X"]
        y = data[client][split]["y"]

        X_flat = X_seq.reshape(X_seq.shape[0], WINDOW * N_FEATURES)

        assert X_flat.shape == (X_seq.shape[0], WINDOW * N_FEATURES)

        tabular_data[client][split] = {
            "X": X_flat,
            "y": y,
            "target_dates": data[client][split]["target_dates"],
            "site_ids": data[client][split]["site_ids"],
        }

print("Tabular conversion completed.")

tabular_shape_rows = []

for client in ACTIVE_CLIENTS:
    for split in SPLITS:
        X_flat = tabular_data[client][split]["X"]
        y = tabular_data[client][split]["y"]

        tabular_shape_rows.append({
            "Client": client,
            "Split": split,
            "X_flat_shape": str(X_flat.shape),
            "y_shape": str(y.shape),
        })

tabular_shape_df = pd.DataFrame(tabular_shape_rows)

display(tabular_shape_df)

save_table(tabular_shape_df, "02_tabular_shape_verification", directory=AUDIT_DIR)

Tabular conversion completed.


,Client,Split,X_flat_shape,y_shape
0,England,train,"(76530, 96)","(76530,)"
1,England,val,"(8490, 96)","(8490,)"
2,England,test,"(11942, 96)","(11942,)"
3,Ireland,train,"(27778, 96)","(27778,)"
4,Ireland,val,"(4296, 96)","(4296,)"
5,Ireland,test,"(5887, 96)","(5887,)"
6,USA,train,"(30222, 96)","(30222,)"
7,USA,val,"(1770, 96)","(1770,)"
8,USA,test,"(1979, 96)","(1979,)"


✓ saved table: 02_tabular_shape_verification.csv


PosixPath('/kaggle/working/classical_ml_baselines/audit/02_tabular_shape_verification.csv')

## Section 3 — Class distribution audit

Before training classical ML models, the target distribution is checked again. This is necessary because accuracy alone can be misleading when one or two classes dominate the data.

In [9]:
# 3.1 — Class distribution per client and split

class_dist_rows = []

for client in ACTIVE_CLIENTS:
    for split in SPLITS:
        y = tabular_data[client][split]["y"]
        total = len(y)

        counts = pd.Series(y).value_counts().reindex(range(N_CLASSES), fill_value=0)

        row = {
            "Client": client,
            "Split": split,
            "N": total,
        }

        for class_id, class_name in enumerate(CLASS_NAMES):
            row[f"{class_name}_count"] = int(counts[class_id])
            row[f"{class_name}_pct"] = round(100 * counts[class_id] / total, 2) if total > 0 else 0.0

        row["Present_classes"] = ", ".join(
            CLASS_NAMES[class_id] for class_id in range(N_CLASSES)
            if counts[class_id] > 0
        )

        class_dist_rows.append(row)

class_distribution_df = pd.DataFrame(class_dist_rows)

display(class_distribution_df)

save_table(class_distribution_df, "03_class_distribution_audit", directory=AUDIT_DIR)

,Client,Split,N,Excellent_count,Excellent_pct,Good_count,Good_pct,Fair_count,Fair_pct,Marginal_count,Marginal_pct,Poor_count,Poor_pct,Present_classes
0,England,train,76530,14591,19.07,28485,37.22,13215,17.27,16360,21.38,3879,5.07,"Excellent, Good, Fair, Marginal, Poor"
1,England,val,8490,2665,31.39,4114,48.46,1147,13.51,526,6.20,38,0.45,"Excellent, Good, Fair, Marginal, Poor"
2,England,test,11942,4162,34.85,5832,48.84,1360,11.39,554,4.64,34,0.28,"Excellent, Good, Fair, Marginal, Poor"
3,Ireland,train,27778,23998,86.39,2975,10.71,485,1.75,96,0.35,224,0.81,"Excellent, Good, Fair, Marginal, Poor"
4,Ireland,val,4296,3706,86.27,576,13.41,13,0.30,1,0.02,0,0.00,"Excellent, Good, Fair, Marginal"
5,Ireland,test,5887,5186,88.09,688,11.69,13,0.22,0,0.00,0,0.00,"Excellent, Good, Fair"
6,USA,train,30222,28012,92.69,2089,6.91,108,0.36,13,0.04,0,0.00,"Excellent, Good, Fair, Marginal"
7,USA,val,1770,1641,92.71,128,7.23,1,0.06,0,0.00,0,0.00,"Excellent, Good, Fair"
8,USA,test,1979,1771,89.49,208,10.51,0,0.00,0,0.00,0,0.00,"Excellent, Good"


✓ saved table: 03_class_distribution_audit.csv


PosixPath('/kaggle/working/classical_ml_baselines/audit/03_class_distribution_audit.csv')

## Section 4 — Evaluation helpers

The following functions are used for all classical ML models. They compute the same main metrics used in the GRU and LSTM notebooks:

- accuracy,
- balanced accuracy,
- macro-F1 over all five classes,
- macro-F1 over classes present in the test set,
- weighted-F1,
- per-class precision, recall, and F1,
- confusion matrix.

The present-class macro-F1 is especially important for Ireland and USA because not all five classes are present in their test sets.

In [10]:
# 4.1 — Metric helper functions

def compute_full_metrics(y_true, y_pred):
    """
    Compute main classification metrics.

    Both all-class and present-class macro-F1 are reported.
    All-class macro-F1 uses the full five-class label set.
    Present-class macro-F1 uses only classes that appear in y_true.
    """
    present_labels = sorted(np.unique(y_true).tolist())

    metrics = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),

        "macro_f1_all_classes": f1_score(
            y_true,
            y_pred,
            labels=list(range(N_CLASSES)),
            average="macro",
            zero_division=0,
        ),

        "macro_f1_present_classes": f1_score(
            y_true,
            y_pred,
            labels=present_labels,
            average="macro",
            zero_division=0,
        ),

        "weighted_f1": f1_score(
            y_true,
            y_pred,
            labels=list(range(N_CLASSES)),
            average="weighted",
            zero_division=0,
        ),

        "present_classes": ", ".join(INT_TO_CLASS[i] for i in present_labels),
    }

    return metrics


def make_summary_row(model_name, strategy, train_client, eval_client, split, y_true, y_pred):
    """Create one metric summary row without runtime columns."""
    metrics = compute_full_metrics(y_true, y_pred)

    row = {
        "Model": model_name,
        "Strategy": strategy,
        "Train_client": train_client,
        "Eval_client": eval_client,
        "Split": split,
        "N": len(y_true),

        "Accuracy": round(metrics["accuracy"], 4),
        "Balanced_accuracy": round(metrics["balanced_accuracy"], 4),
        "Macro_F1_all_classes": round(metrics["macro_f1_all_classes"], 4),
        "Macro_F1_present_classes": round(metrics["macro_f1_present_classes"], 4),
        "Weighted_F1": round(metrics["weighted_f1"], 4),

        "Present_classes": metrics["present_classes"],
    }

    return row


def make_per_class_rows(model_name, strategy, train_client, eval_client, split, y_true, y_pred):
    """Create per-class precision/recall/F1 rows."""
    report = classification_report(
        y_true,
        y_pred,
        labels=list(range(N_CLASSES)),
        target_names=CLASS_NAMES,
        zero_division=0,
        output_dict=True,
    )

    rows = []

    for class_id, class_name in enumerate(CLASS_NAMES):
        rows.append({
            "Model": model_name,
            "Strategy": strategy,
            "Train_client": train_client,
            "Eval_client": eval_client,
            "Split": split,
            "Class_id": class_id,
            "Class": class_name,
            "Support": int(report[class_name]["support"]),
            "Precision": round(report[class_name]["precision"], 4),
            "Recall": round(report[class_name]["recall"], 4),
            "F1": round(report[class_name]["f1-score"], 4),
        })

    return rows


def make_confusion_matrix_df(y_true, y_pred):
    """Return a labeled confusion matrix DataFrame."""
    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(range(N_CLASSES)),
    )

    cm_df = pd.DataFrame(
        cm,
        index=[f"true_{name}" for name in CLASS_NAMES],
        columns=[f"pred_{name}" for name in CLASS_NAMES],
    )

    return cm_df

In [11]:
# 4.2 — Metric helper smoke test

_smoke_y_true = np.array([0, 0, 1, 1, 2, 2])
_smoke_y_pred = np.array([0, 1, 1, 1, 2, 0])

_smoke_metrics = compute_full_metrics(_smoke_y_true, _smoke_y_pred)

print("Metric helper smoke test:")
display(pd.DataFrame([_smoke_metrics]).round(4))

print("Confusion matrix smoke test:")
display(make_confusion_matrix_df(_smoke_y_true, _smoke_y_pred))

print("✓ Metric helpers are working.")

Metric helper smoke test:


,accuracy,balanced_accuracy,macro_f1_all_classes,macro_f1_present_classes,weighted_f1,present_classes
0,0.6667,0.6667,0.3933,0.6556,0.6556,"Excellent, Good, Fair"


Confusion matrix smoke test:


,pred_Excellent,pred_Good,pred_Fair,pred_Marginal,pred_Poor
true_Excellent,1,1,0,0,0
true_Good,0,2,0,0,0
true_Fair,1,0,1,0,0
true_Marginal,0,0,0,0,0
true_Poor,0,0,0,0,0


✓ Metric helpers are working.


## Section 5 — Common training and evaluation helper

This section defines a common helper used by all classical ML models. The helper trains one model locally for each client and evaluates it on the validation and test splits.

The notebook reports the same metrics used in previous GRU/LSTM experiments: accuracy, balanced accuracy, macro-F1 over all classes, macro-F1 over present classes, weighted-F1, per-class metrics, and confusion matrices.

In [12]:
# 5.1 — Import classical ML models

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.base import clone
import joblib

from xgboost import XGBClassifier
from catboost import CatBoostClassifier

print("Classical ML models imported:")
print("- Logistic Regression")
print("- Random Forest")
print("- XGBoost")
print("- CatBoost")

Classical ML models imported:
- Logistic Regression
- Random Forest
- XGBoost
- CatBoost


In [13]:
# 5.2 — Prediction saving helper

def save_predictions_table(model_name, strategy, train_client, eval_client, split, y_true, y_pred, target_dates, site_ids):
    """
    Save predictions for one model/client/split.

    All arrays are flattened to 1D because some libraries, especially CatBoost,
    may return predictions with shape (N, 1) instead of (N,).
    """
    y_true = np.asarray(y_true).reshape(-1).astype(int)
    y_pred = np.asarray(y_pred).reshape(-1).astype(int)
    target_dates = np.asarray(target_dates).reshape(-1)
    site_ids = np.asarray(site_ids).reshape(-1)

    assert len(y_true) == len(y_pred) == len(target_dates) == len(site_ids), (
        f"Length mismatch: y_true={len(y_true)}, y_pred={len(y_pred)}, "
        f"target_dates={len(target_dates)}, site_ids={len(site_ids)}"
    )

    pred_df = pd.DataFrame({
        "Model": model_name,
        "Strategy": strategy,
        "Train_client": train_client,
        "Eval_client": eval_client,
        "Split": split,
        "site_id": site_ids,
        "target_date": target_dates,
        "y_true": y_true,
        "y_true_class": [INT_TO_CLASS[int(v)] for v in y_true],
        "y_pred": y_pred,
        "y_pred_class": [INT_TO_CLASS[int(v)] for v in y_pred],
        "correct": y_true == y_pred,
    })

    safe_name = f"{strategy}_{model_name}_{train_client}_to_{eval_client}_{split}".lower()
    safe_name = safe_name.replace(" ", "_").replace("/", "_")

    path = PRED_DIR / f"{safe_name}_predictions.csv"
    pred_df.to_csv(path, index=False)

    return path

In [14]:
# 5.3 — Local model runner

def run_local_model(model_name, model):
    """
    Train one classical ML model locally for each active client.

    For each client:
    - fit on train split,
    - evaluate on validation split,
    - evaluate on test split,
    - save predictions and confusion matrices,
    - return summary and per-class metrics.
    """

    summary_tables = []
    per_class_tables = []
    confusion_matrices = {}
    fitted_models = {}

    print("\n" + "=" * 100)
    print(f"LOCAL BASELINE — {model_name}")
    print("=" * 100)

    for client in ACTIVE_CLIENTS:
        print("\n" + "-" * 80)
        print(f"Training {model_name} for client: {client}")
        print("-" * 80)

        X_train = tabular_data[client]["train"]["X"]
        y_train = tabular_data[client]["train"]["y"]

        fitted_model = clone(model)

        fit_start = time.time()

        if model_name == "XGBoost":
            sample_weight = compute_sample_weight(class_weight="balanced", y=y_train)
            fitted_model.fit(X_train, y_train, sample_weight=sample_weight)
        else:
            fitted_model.fit(X_train, y_train)

        fit_time = time.time() - fit_start

        fitted_models[client] = fitted_model

        for split in ["val", "test"]:
            X_eval = tabular_data[client][split]["X"]
            y_eval = tabular_data[client][split]["y"]

            pred_start = time.time()
            y_pred = fitted_model.predict(X_eval)
            predict_time = time.time() - pred_start

            # Important: flatten predictions for all models
            y_pred = np.asarray(y_pred).reshape(-1).astype(int)

            summary_row = make_summary_row(
                model_name=model_name,
                strategy="Local",
                train_client=client,
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
            )

            summary_tables.append(pd.DataFrame([summary_row]))

            per_class_rows = make_per_class_rows(
                model_name=model_name,
                strategy="Local",
                train_client=client,
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
            )

            per_class_tables.append(pd.DataFrame(per_class_rows))

            cm_df = make_confusion_matrix_df(y_eval, y_pred)
            confusion_matrices[(client, split)] = cm_df

            cm_name = f"local_{model_name}_{client}_{split}_confusion_matrix".lower()
            cm_name = cm_name.replace(" ", "_").replace("/", "_")
            save_table(cm_df.reset_index().rename(columns={"index": "True_class"}), cm_name, directory=METRIC_DIR)

            save_predictions_table(
                model_name=model_name,
                strategy="Local",
                train_client=client,
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
                target_dates=tabular_data[client][split]["target_dates"],
                site_ids=tabular_data[client][split]["site_ids"],
            )

        model_path = MODEL_DIR / f"local_{model_name}_{client}.joblib".lower().replace(" ", "_")
        joblib.dump(fitted_model, model_path)

        print(f"Finished {model_name} for {client}.")

    summary_df = pd.concat(summary_tables, ignore_index=True)
    per_class_df = pd.concat(per_class_tables, ignore_index=True)

    print("\nSummary metrics:")
    display(summary_df)

    print("\nTest metrics only:")
    display(summary_df[summary_df["Split"] == "test"].sort_values(
        by=["Eval_client", "Macro_F1_present_classes"],
        ascending=[True, False]
    ))

    return summary_df, per_class_df, confusion_matrices, fitted_models

## Section 6 — Local Logistic Regression baseline

Logistic Regression is used as a simple linear baseline. It helps us check whether the flattened 12-month water-quality features contain enough separable information without using complex nonlinear models.

Because the class distribution is imbalanced, `class_weight="balanced"` is used.

In [15]:
# 6.1 — Define and run Logistic Regression

logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=700,
    solver="lbfgs",
    random_state=SEED,
    n_jobs=-1,
)

logreg_summary_df, logreg_per_class_df, logreg_cms, logreg_models = run_local_model(
    model_name="Logistic_Regression",
    model=logistic_model,
)

_ = save_table(logreg_summary_df, "local_logistic_regression_summary_metrics", directory=METRIC_DIR)
_ = save_table(logreg_per_class_df, "local_logistic_regression_per_class_metrics", directory=METRIC_DIR)


LOCAL BASELINE — Logistic_Regression

--------------------------------------------------------------------------------
Training Logistic_Regression for client: England
--------------------------------------------------------------------------------
✓ saved table: local_logistic_regression_england_val_confusion_matrix.csv
✓ saved table: local_logistic_regression_england_test_confusion_matrix.csv
Finished Logistic_Regression for England.

--------------------------------------------------------------------------------
Training Logistic_Regression for client: Ireland
--------------------------------------------------------------------------------
✓ saved table: local_logistic_regression_ireland_val_confusion_matrix.csv
✓ saved table: local_logistic_regression_ireland_test_confusion_matrix.csv
Finished Logistic_Regression for Ireland.

--------------------------------------------------------------------------------
Training Logistic_Regression for client: USA
-----------------------------

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,Logistic_Regression,Local,England,England,val,8490,0.7450,0.7051,0.7085,0.7085,0.7432,"Excellent, Good, Fair, Marginal, Poor"
1,Logistic_Regression,Local,England,England,test,11942,0.7764,0.6302,0.6567,0.6567,0.7736,"Excellent, Good, Fair, Marginal, Poor"
2,Logistic_Regression,Local,Ireland,Ireland,val,4296,0.6848,0.3765,0.2511,0.3138,0.7373,"Excellent, Good, Fair, Marginal"
3,Logistic_Regression,Local,Ireland,Ireland,test,5887,0.7297,0.4799,0.2471,0.4118,0.7778,"Excellent, Good, Fair"
4,Logistic_Regression,Local,USA,USA,val,1770,0.9305,0.5675,0.3246,0.5410,0.9426,"Excellent, Good, Fair"
5,Logistic_Regression,Local,USA,USA,test,1979,0.8863,0.7180,0.2969,0.7421,0.9019,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,Logistic_Regression,Local,England,England,test,11942,0.7764,0.6302,0.6567,0.6567,0.7736,"Excellent, Good, Fair, Marginal, Poor"
3,Logistic_Regression,Local,Ireland,Ireland,test,5887,0.7297,0.4799,0.2471,0.4118,0.7778,"Excellent, Good, Fair"
5,Logistic_Regression,Local,USA,USA,test,1979,0.8863,0.7180,0.2969,0.7421,0.9019,"Excellent, Good"


✓ saved table: local_logistic_regression_summary_metrics.csv
✓ saved table: local_logistic_regression_per_class_metrics.csv


### Initial interpretation of the local Logistic Regression baseline

The local Logistic Regression baseline provides a simple linear reference for each client. England obtains the most balanced result among the three clients because all five classes are present in its test set. The model reaches a test macro-F1 over present classes of 0.6567, which shows that the flattened 12-month features contain useful predictive information, although the result remains below the previous local GRU baseline.

Ireland remains more difficult. Although the test accuracy is 0.7297, the macro-F1 over present classes is only 0.4120, indicating that the model struggles with class-balanced prediction under strong class imbalance.

For USA, Logistic Regression obtains a relatively strong macro-F1 over present classes of 0.7421. However, this result must be interpreted carefully because the USA test set contains only Excellent and Good classes. Therefore, the high accuracy does not mean that the model can distinguish all five water-quality categories.

Overall, Logistic Regression is a useful baseline, but it does not yet outperform the recurrent GRU/LSTM baselines. The next step is to evaluate nonlinear tree-based models, starting with Random Forest, to test whether they can better capture nonlinear relationships in the flattened sequence features.

In [16]:
# 6.2 — Prediction saving helper

def save_predictions_table(model_name, strategy, train_client, eval_client, split, y_true, y_pred, target_dates, site_ids):
    """
    Save predictions for one model/client/split.
    """
    pred_df = pd.DataFrame({
        "Model": model_name,
        "Strategy": strategy,
        "Train_client": train_client,
        "Eval_client": eval_client,
        "Split": split,
        "site_id": site_ids,
        "target_date": target_dates,
        "y_true": y_true,
        "y_true_class": [INT_TO_CLASS[int(v)] for v in y_true],
        "y_pred": y_pred,
        "y_pred_class": [INT_TO_CLASS[int(v)] for v in y_pred],
        "correct": y_true == y_pred,
    })

    safe_name = f"{strategy}_{model_name}_{train_client}_to_{eval_client}_{split}".lower()
    safe_name = safe_name.replace(" ", "_").replace("/", "_")

    path = PRED_DIR / f"{safe_name}_predictions.csv"
    pred_df.to_csv(path, index=False)

    return path

## Section 7 — Local Random Forest baseline

Random Forest is a nonlinear tree-based ensemble model. It can capture interactions between water-quality parameters without requiring a neural sequence architecture.

In this experiment, one Random Forest model is trained separately for each client. The model uses the flattened 12-month sequence features, where each sample contains 96 input features.

The model uses `class_weight="balanced_subsample"` to reduce the effect of class imbalance during tree construction.

In [17]:
# 7.1 — Define and run Random Forest

random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)

rf_summary_df, rf_per_class_df, rf_cms, rf_models = run_local_model(
    model_name="Random_Forest",
    model=random_forest_model,
)

_ = save_table(rf_summary_df, "local_random_forest_summary_metrics", directory=METRIC_DIR)
_ = save_table(rf_per_class_df, "local_random_forest_per_class_metrics", directory=METRIC_DIR)


LOCAL BASELINE — Random_Forest

--------------------------------------------------------------------------------
Training Random_Forest for client: England
--------------------------------------------------------------------------------
✓ saved table: local_random_forest_england_val_confusion_matrix.csv
✓ saved table: local_random_forest_england_test_confusion_matrix.csv
Finished Random_Forest for England.

--------------------------------------------------------------------------------
Training Random_Forest for client: Ireland
--------------------------------------------------------------------------------
✓ saved table: local_random_forest_ireland_val_confusion_matrix.csv
✓ saved table: local_random_forest_ireland_test_confusion_matrix.csv
Finished Random_Forest for Ireland.

--------------------------------------------------------------------------------
Training Random_Forest for client: USA
--------------------------------------------------------------------------------
✓ saved 

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,Random_Forest,Local,England,England,val,8490,0.8011,0.6983,0.7313,0.7313,0.8003,"Excellent, Good, Fair, Marginal, Poor"
1,Random_Forest,Local,England,England,test,11942,0.8261,0.6472,0.6832,0.6832,0.8240,"Excellent, Good, Fair, Marginal, Poor"
2,Random_Forest,Local,Ireland,Ireland,val,4296,0.9004,0.3493,0.2957,0.3696,0.8869,"Excellent, Good, Fair, Marginal"
3,Random_Forest,Local,Ireland,Ireland,test,5887,0.9176,0.4716,0.3018,0.5030,0.9061,"Excellent, Good, Fair"
4,Random_Forest,Local,USA,USA,val,1770,0.9610,0.5688,0.3423,0.5704,0.9606,"Excellent, Good, Fair"
5,Random_Forest,Local,USA,USA,test,1979,0.9338,0.7254,0.3117,0.7792,0.9251,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,Random_Forest,Local,England,England,test,11942,0.8261,0.6472,0.6832,0.6832,0.8240,"Excellent, Good, Fair, Marginal, Poor"
3,Random_Forest,Local,Ireland,Ireland,test,5887,0.9176,0.4716,0.3018,0.5030,0.9061,"Excellent, Good, Fair"
5,Random_Forest,Local,USA,USA,test,1979,0.9338,0.7254,0.3117,0.7792,0.9251,"Excellent, Good"


✓ saved table: local_random_forest_summary_metrics.csv
✓ saved table: local_random_forest_per_class_metrics.csv


### Initial interpretation of the local Random Forest baseline

The local Random Forest baseline improves over Logistic Regression for all three clients when using macro-F1 over present classes. This suggests that the flattened 12-month water-quality features contain nonlinear relationships that are better captured by tree-based models than by a linear classifier.

For England, Random Forest reaches a test macro-F1 over present classes of 0.6832. This is stronger than Logistic Regression, although still slightly below the previous local GRU baseline. Since England contains all five classes in the test set, this result is a meaningful full-class evaluation.

For Ireland, Random Forest gives a clear improvement over Logistic Regression, with macro-F1 over present classes increasing to 0.5030. However, the balanced accuracy remains limited, showing that class imbalance and rare classes are still difficult.

For USA, Random Forest obtains the highest score so far among the local classical ML models, with a macro-F1 over present classes of 0.7792. This is competitive with the previous local recurrent baselines. However, the result must be interpreted carefully because the USA test set contains only Excellent and Good classes.

Overall, Random Forest is a strong classical baseline and confirms that nonlinear tabular models are important candidates for this dataset.

## Section 8 — Local XGBoost baseline

XGBoost is a gradient-boosted tree model. Unlike Random Forest, which builds many trees independently, XGBoost builds trees sequentially, where each new tree tries to correct the errors made by the previous trees.

In this experiment, one XGBoost model is trained separately for each client using the same flattened 12-month sequence features. Since the data are imbalanced, balanced sample weights are passed during training.

The purpose of this baseline is to check whether a stronger boosted tree model can improve over Logistic Regression and Random Forest.

In [18]:
# 8.1 — Define and run XGBoost

xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

xgb_summary_df, xgb_per_class_df, xgb_cms, xgb_models = run_local_model(
    model_name="XGBoost",
    model=xgboost_model,
)

_ = save_table(xgb_summary_df, "local_xgboost_summary_metrics", directory=METRIC_DIR)
_ = save_table(xgb_per_class_df, "local_xgboost_per_class_metrics", directory=METRIC_DIR)


LOCAL BASELINE — XGBoost

--------------------------------------------------------------------------------
Training XGBoost for client: England
--------------------------------------------------------------------------------
✓ saved table: local_xgboost_england_val_confusion_matrix.csv
✓ saved table: local_xgboost_england_test_confusion_matrix.csv
Finished XGBoost for England.

--------------------------------------------------------------------------------
Training XGBoost for client: Ireland
--------------------------------------------------------------------------------
✓ saved table: local_xgboost_ireland_val_confusion_matrix.csv
✓ saved table: local_xgboost_ireland_test_confusion_matrix.csv
Finished XGBoost for Ireland.

--------------------------------------------------------------------------------
Training XGBoost for client: USA
--------------------------------------------------------------------------------
✓ saved table: local_xgboost_usa_val_confusion_matrix.csv
✓ saved ta

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Local,England,England,val,8490,0.7898,0.7311,0.7363,0.7363,0.7914,"Excellent, Good, Fair, Marginal, Poor"
1,XGBoost,Local,England,England,test,11942,0.8180,0.7016,0.7145,0.7145,0.8190,"Excellent, Good, Fair, Marginal, Poor"
2,XGBoost,Local,Ireland,Ireland,val,4296,0.8298,0.4117,0.2916,0.3645,0.8485,"Excellent, Good, Fair, Marginal"
3,XGBoost,Local,Ireland,Ireland,test,5887,0.8648,0.5550,0.2997,0.4995,0.8784,"Excellent, Good, Fair"
4,XGBoost,Local,USA,USA,val,1770,0.9401,0.6213,0.3318,0.5530,0.9464,"Excellent, Good, Fair"
5,XGBoost,Local,USA,USA,test,1979,0.9096,0.8307,0.3156,0.7889,0.9150,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,XGBoost,Local,England,England,test,11942,0.8180,0.7016,0.7145,0.7145,0.8190,"Excellent, Good, Fair, Marginal, Poor"
3,XGBoost,Local,Ireland,Ireland,test,5887,0.8648,0.5550,0.2997,0.4995,0.8784,"Excellent, Good, Fair"
5,XGBoost,Local,USA,USA,test,1979,0.9096,0.8307,0.3156,0.7889,0.9150,"Excellent, Good"


✓ saved table: local_xgboost_summary_metrics.csv
✓ saved table: local_xgboost_per_class_metrics.csv


### Initial interpretation of the local XGBoost baseline

The local XGBoost baseline gives the strongest classical ML result so far for England and USA, and a result very close to Random Forest for Ireland. This suggests that boosted tree models are well suited to the flattened 12-month water-quality features.

For England, XGBoost reaches a test macro-F1 over present classes of 0.7145. Since all five classes are present in the England test set, this is a strong full-class result and is higher than the previous local GRU baseline.

For Ireland, XGBoost obtains a macro-F1 over present classes of 0.4995, which is close to Random Forest. Although its accuracy is lower than Random Forest, its balanced accuracy is higher, suggesting better class-recall balance. This needs to be checked using the per-class metrics and confusion matrix.

For USA, XGBoost reaches a macro-F1 over present classes of 0.7889, which is the strongest local result so far. However, the USA test set contains only Excellent and Good classes, so the result should be interpreted as strong binary-like performance rather than full five-class classification.

Overall, XGBoost is currently the strongest local classical ML model for England and USA, and it confirms that tree-based boosting is a very competitive approach for this dataset.

## Section 9 — Local CatBoost baseline

CatBoost is a gradient-boosted tree model designed for tabular data. Like XGBoost, it builds trees sequentially, where later trees correct previous errors.

In this experiment, one CatBoost model is trained separately for each client using the same flattened 12-month sequence features.

The model uses `auto_class_weights="Balanced"` to reduce the effect of class imbalance. This is important because Ireland and USA are strongly dominated by Excellent and Good classes.

In [19]:
# 9.1 — Define and run CatBoost

catboost_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)

cat_summary_df, cat_per_class_df, cat_cms, cat_models = run_local_model(
    model_name="CatBoost",
    model=catboost_model,
)

_ = save_table(cat_summary_df, "local_catboost_summary_metrics", directory=METRIC_DIR)
_ = save_table(cat_per_class_df, "local_catboost_per_class_metrics", directory=METRIC_DIR)


LOCAL BASELINE — CatBoost

--------------------------------------------------------------------------------
Training CatBoost for client: England
--------------------------------------------------------------------------------
✓ saved table: local_catboost_england_val_confusion_matrix.csv
✓ saved table: local_catboost_england_test_confusion_matrix.csv
Finished CatBoost for England.

--------------------------------------------------------------------------------
Training CatBoost for client: Ireland
--------------------------------------------------------------------------------
✓ saved table: local_catboost_ireland_val_confusion_matrix.csv
✓ saved table: local_catboost_ireland_test_confusion_matrix.csv
Finished CatBoost for Ireland.

--------------------------------------------------------------------------------
Training CatBoost for client: USA
--------------------------------------------------------------------------------
✓ saved table: local_catboost_usa_val_confusion_matrix.csv

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,CatBoost,Local,England,England,val,8490,0.7803,0.7435,0.7345,0.7345,0.7820,"Excellent, Good, Fair, Marginal, Poor"
1,CatBoost,Local,England,England,test,11942,0.8087,0.6728,0.6973,0.6973,0.8096,"Excellent, Good, Fair, Marginal, Poor"
2,CatBoost,Local,Ireland,Ireland,val,4296,0.8126,0.4096,0.2853,0.3566,0.8355,"Excellent, Good, Fair, Marginal"
3,CatBoost,Local,Ireland,Ireland,test,5887,0.8558,0.5549,0.2960,0.4933,0.8718,"Excellent, Good, Fair"
4,CatBoost,Local,USA,USA,val,1770,0.9384,0.6207,0.3308,0.5513,0.9452,"Excellent, Good, Fair"
5,CatBoost,Local,USA,USA,test,1979,0.9050,0.8366,0.3139,0.7848,0.9118,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,CatBoost,Local,England,England,test,11942,0.8087,0.6728,0.6973,0.6973,0.8096,"Excellent, Good, Fair, Marginal, Poor"
3,CatBoost,Local,Ireland,Ireland,test,5887,0.8558,0.5549,0.2960,0.4933,0.8718,"Excellent, Good, Fair"
5,CatBoost,Local,USA,USA,test,1979,0.9050,0.8366,0.3139,0.7848,0.9118,"Excellent, Good"


✓ saved table: local_catboost_summary_metrics.csv
✓ saved table: local_catboost_per_class_metrics.csv


### Initial interpretation of the local CatBoost baseline

The local CatBoost baseline performs strongly across the three active clients. It improves clearly over Logistic Regression and remains close to XGBoost and Random Forest.

For England, CatBoost reaches a test macro-F1 over present classes of 0.6973. This is a strong result because England contains all five classes in the test set, although XGBoost remains stronger in this local setting.

For Ireland, CatBoost obtains a macro-F1 over present classes of 0.4933. This is close to both XGBoost and Random Forest, showing that tree-based models are more suitable than the linear baseline, but the client remains difficult because of class imbalance and rare classes.

For USA, CatBoost reaches a macro-F1 over present classes of 0.7848 and the highest balanced accuracy among the local classical ML models. However, the USA test set contains only Excellent and Good classes, so this result should be interpreted as strong performance on the present classes rather than complete five-class classification.

Overall, CatBoost is a competitive classical ML baseline, but XGBoost remains the strongest local classical model for England and USA in the current experiment.

## Section 10 — Local classical ML comparison

This section combines the local results from Logistic Regression, Random Forest, XGBoost, and CatBoost.

The main comparison metric is macro-F1 over present classes, because it is more informative than accuracy when some clients are dominated by Excellent and Good classes or when some classes are absent from the test set.

In [20]:
# 10.1 — Combine local classical ML results

local_classical_summary_df = pd.concat(
    [
        logreg_summary_df,
        rf_summary_df,
        xgb_summary_df,
        cat_summary_df,
    ],
    ignore_index=True,
)

local_classical_per_class_df = pd.concat(
    [
        logreg_per_class_df,
        rf_per_class_df,
        xgb_per_class_df,
        cat_per_class_df,
    ],
    ignore_index=True,
)

local_classical_test_df = local_classical_summary_df[
    local_classical_summary_df["Split"] == "test"
].copy()

local_classical_test_df = local_classical_test_df.sort_values(
    by=["Eval_client", "Macro_F1_present_classes"],
    ascending=[True, False],
).reset_index(drop=True)

print("Local classical ML test results ranked by client and Macro-F1 present:")
display(local_classical_test_df)

_ = save_table(local_classical_summary_df, "local_classical_ml_all_summary_metrics", directory=METRIC_DIR)
_ = save_table(local_classical_per_class_df, "local_classical_ml_all_per_class_metrics", directory=METRIC_DIR)
_ = save_table(local_classical_test_df, "local_classical_ml_test_results_ranked", directory=METRIC_DIR)

Local classical ML test results ranked by client and Macro-F1 present:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Local,England,England,test,11942,0.8180,0.7016,0.7145,0.7145,0.8190,"Excellent, Good, Fair, Marginal, Poor"
1,CatBoost,Local,England,England,test,11942,0.8087,0.6728,0.6973,0.6973,0.8096,"Excellent, Good, Fair, Marginal, Poor"
2,Random_Forest,Local,England,England,test,11942,0.8261,0.6472,0.6832,0.6832,0.8240,"Excellent, Good, Fair, Marginal, Poor"
3,Logistic_Regression,Local,England,England,test,11942,0.7764,0.6302,0.6567,0.6567,0.7736,"Excellent, Good, Fair, Marginal, Poor"
4,Random_Forest,Local,Ireland,Ireland,test,5887,0.9176,0.4716,0.3018,0.5030,0.9061,"Excellent, Good, Fair"
5,XGBoost,Local,Ireland,Ireland,test,5887,0.8648,0.5550,0.2997,0.4995,0.8784,"Excellent, Good, Fair"
6,CatBoost,Local,Ireland,Ireland,test,5887,0.8558,0.5549,0.2960,0.4933,0.8718,"Excellent, Good, Fair"
7,Logistic_Regression,Local,Ireland,Ireland,test,5887,0.7297,0.4799,0.2471,0.4118,0.7778,"Excellent, Good, Fair"
8,XGBoost,Local,USA,USA,test,1979,0.9096,0.8307,0.3156,0.7889,0.9150,"Excellent, Good"
9,CatBoost,Local,USA,USA,test,1979,0.9050,0.8366,0.3139,0.7848,0.9118,"Excellent, Good"


✓ saved table: local_classical_ml_all_summary_metrics.csv
✓ saved table: local_classical_ml_all_per_class_metrics.csv
✓ saved table: local_classical_ml_test_results_ranked.csv


In [21]:
# 10.2 — Best local classical ML model per client

best_local_classical_df = (
    local_classical_test_df
    .sort_values(["Eval_client", "Macro_F1_present_classes"], ascending=[True, False])
    .groupby("Eval_client", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best local classical ML model per client:")
display(best_local_classical_df)

_ = save_table(best_local_classical_df, "best_local_classical_ml_model_per_client", directory=METRIC_DIR)

Best local classical ML model per client:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Local,England,England,test,11942,0.8180,0.7016,0.7145,0.7145,0.8190,"Excellent, Good, Fair, Marginal, Poor"
1,Random_Forest,Local,Ireland,Ireland,test,5887,0.9176,0.4716,0.3018,0.5030,0.9061,"Excellent, Good, Fair"
2,XGBoost,Local,USA,USA,test,1979,0.9096,0.8307,0.3156,0.7889,0.9150,"Excellent, Good"


✓ saved table: best_local_classical_ml_model_per_client.csv


In [22]:
# 10.3 — Compact comparison table: Macro-F1 present classes

macro_f1_pivot_df = local_classical_test_df.pivot_table(
    index="Eval_client",
    columns="Model",
    values="Macro_F1_present_classes",
    aggfunc="first",
)

macro_f1_pivot_df = macro_f1_pivot_df.reindex(ACTIVE_CLIENTS)

print("Local classical ML comparison using Macro-F1 over present classes:")
display(macro_f1_pivot_df.round(4))

_ = save_table(
    macro_f1_pivot_df.reset_index(),
    "local_classical_ml_macro_f1_present_pivot",
    directory=METRIC_DIR,
)

Local classical ML comparison using Macro-F1 over present classes:


Model,CatBoost,Logistic_Regression,Random_Forest,XGBoost
Eval_client,,,,
England,0.6973,0.6567,0.6832,0.7145
Ireland,0.4933,0.4118,0.5030,0.4995
USA,0.7848,0.7421,0.7792,0.7889


✓ saved table: local_classical_ml_macro_f1_present_pivot.csv


### Interpretation of local classical ML baselines

The local classical ML results show that nonlinear tree-based models are more effective than Logistic Regression for the sequence-to-class water-quality forecasting task. Logistic Regression provides a useful linear baseline, but it is consistently weaker than Random Forest, XGBoost, and CatBoost across the three clients.

For England, XGBoost gives the best local classical ML result, with a test macro-F1 over present classes of 0.7145. This is particularly important because England contains all five water-quality classes in the test set, making it the most complete client for class-balanced evaluation.

For Ireland, Random Forest obtains the highest macro-F1 over present classes, although XGBoost and CatBoost are very close. The results should be interpreted carefully because Ireland is strongly imbalanced and the test set contains only Excellent, Good, and Fair classes.

For USA, XGBoost gives the highest macro-F1 over present classes, while CatBoost gives the highest balanced accuracy. However, the USA test set contains only Excellent and Good classes, so this result reflects strong performance on the present classes rather than full five-class prediction.

Overall, the local classical ML benchmark confirms that tree-based models, especially XGBoost, are highly competitive for this dataset. These results justify comparing classical ML models against GRU/LSTM baselines and later using SHAP to interpret the strongest tree-based models.

## Phase 3 — Centralized classical ML baselines

In this phase, we train one classical machine learning model using the pooled training data from England, Ireland, and USA.

This setting is called centralized because the training data from all clients are combined into one training set. It is not privacy-preserving, so it is not federated learning. However, it provides an important reference baseline for comparison with local and federated models.

The evaluation remains client-specific: after training the centralized model, it is tested separately on England, Ireland, and USA validation/test splits.

## Section 11 — Build centralized training data

This section concatenates the training sequences from England, Ireland, and USA into one pooled training set. Validation and test evaluation remain separated by client, so the centralized models can still be compared fairly with the local models on each country.


In [23]:
# 11.1 — Build pooled centralized training data

X_train_central = np.concatenate(
    [tabular_data[client]["train"]["X"] for client in ACTIVE_CLIENTS],
    axis=0,
)

y_train_central = np.concatenate(
    [tabular_data[client]["train"]["y"] for client in ACTIVE_CLIENTS],
    axis=0,
)

print("Centralized training data prepared.")
print(f"X_train_central shape: {X_train_central.shape}")
print(f"y_train_central shape: {y_train_central.shape}")

Centralized training data prepared.
X_train_central shape: (134530, 96)
y_train_central shape: (134530,)


In [24]:
# 11.2 — Centralized training data composition

central_train_rows = []

for client in ACTIVE_CLIENTS:
    y_client = tabular_data[client]["train"]["y"]
    counts = pd.Series(y_client).value_counts().reindex(range(N_CLASSES), fill_value=0)

    row = {
        "Client": client,
        "Train_sequences": len(y_client),
    }

    for class_id, class_name in enumerate(CLASS_NAMES):
        row[f"{class_name}_count"] = int(counts[class_id])
        row[f"{class_name}_pct"] = round(100 * counts[class_id] / len(y_client), 2)

    central_train_rows.append(row)

pooled_counts = pd.Series(y_train_central).value_counts().reindex(range(N_CLASSES), fill_value=0)

pooled_row = {
    "Client": "Pooled",
    "Train_sequences": len(y_train_central),
}

for class_id, class_name in enumerate(CLASS_NAMES):
    pooled_row[f"{class_name}_count"] = int(pooled_counts[class_id])
    pooled_row[f"{class_name}_pct"] = round(100 * pooled_counts[class_id] / len(y_train_central), 2)

central_train_rows.append(pooled_row)

central_train_distribution_df = pd.DataFrame(central_train_rows)

print("Centralized training class distribution:")
display(central_train_distribution_df)

_ = save_table(
    central_train_distribution_df,
    "centralized_training_class_distribution",
    directory=AUDIT_DIR,
)

Centralized training class distribution:


,Client,Train_sequences,Excellent_count,Excellent_pct,Good_count,Good_pct,Fair_count,Fair_pct,Marginal_count,Marginal_pct,Poor_count,Poor_pct
0,England,76530,14591,19.07,28485,37.22,13215,17.27,16360,21.38,3879,5.07
1,Ireland,27778,23998,86.39,2975,10.71,485,1.75,96,0.35,224,0.81
2,USA,30222,28012,92.69,2089,6.91,108,0.36,13,0.04,0,0.00
3,Pooled,134530,66601,49.51,33549,24.94,13808,10.26,16469,12.24,4103,3.05


✓ saved table: centralized_training_class_distribution.csv


## Section 12 — Centralized training and evaluation helper

The following helper trains one model on the pooled training set and evaluates it separately on each client.

The same metrics are reported as in the local experiments: accuracy, balanced accuracy, macro-F1 over all classes, macro-F1 over present classes, weighted-F1, per-class metrics, and confusion matrices.

In [25]:
# 12.1 — Centralized model runner

def run_centralized_model(model_name, model):
    """
    Train one classical ML model on pooled training data from all active clients.

    Evaluation remains separated by client:
    - England validation/test
    - Ireland validation/test
    - USA validation/test
    """

    summary_tables = []
    per_class_tables = []
    confusion_matrices = {}

    print("\n" + "=" * 100)
    print(f"CENTRALIZED BASELINE — {model_name}")
    print("=" * 100)

    fitted_model = clone(model)

    print("\nTraining on pooled data: England + Ireland + USA")
    print(f"Pooled train shape: {X_train_central.shape}")

    if model_name == "XGBoost":
        sample_weight = compute_sample_weight(class_weight="balanced", y=y_train_central)
        fitted_model.fit(X_train_central, y_train_central, sample_weight=sample_weight)
    else:
        fitted_model.fit(X_train_central, y_train_central)

    print(f"Finished centralized training for {model_name}.")

    for client in ACTIVE_CLIENTS:
        print("\n" + "-" * 80)
        print(f"Evaluating centralized {model_name} on client: {client}")
        print("-" * 80)

        for split in ["val", "test"]:
            X_eval = tabular_data[client][split]["X"]
            y_eval = tabular_data[client][split]["y"]

            y_pred = fitted_model.predict(X_eval)
            y_pred = np.asarray(y_pred).reshape(-1).astype(int)

            summary_row = make_summary_row(
                model_name=model_name,
                strategy="Centralized",
                train_client="Pooled",
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
            )

            summary_tables.append(pd.DataFrame([summary_row]))

            per_class_rows = make_per_class_rows(
                model_name=model_name,
                strategy="Centralized",
                train_client="Pooled",
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
            )

            per_class_tables.append(pd.DataFrame(per_class_rows))

            cm_df = make_confusion_matrix_df(y_eval, y_pred)
            confusion_matrices[(client, split)] = cm_df

            cm_name = f"centralized_{model_name}_{client}_{split}_confusion_matrix".lower()
            cm_name = cm_name.replace(" ", "_").replace("/", "_")

            _ = save_table(
                cm_df.reset_index().rename(columns={"index": "True_class"}),
                cm_name,
                directory=METRIC_DIR,
            )

            save_predictions_table(
                model_name=model_name,
                strategy="Centralized",
                train_client="Pooled",
                eval_client=client,
                split=split,
                y_true=y_eval,
                y_pred=y_pred,
                target_dates=tabular_data[client][split]["target_dates"],
                site_ids=tabular_data[client][split]["site_ids"],
            )

    model_path = MODEL_DIR / f"centralized_{model_name}.joblib".lower().replace(" ", "_")
    joblib.dump(fitted_model, model_path)

    summary_df = pd.concat(summary_tables, ignore_index=True)
    per_class_df = pd.concat(per_class_tables, ignore_index=True)

    print("\nSummary metrics:")
    display(summary_df)

    print("\nTest metrics only:")
    display(
        summary_df[summary_df["Split"] == "test"].sort_values(
            by=["Eval_client", "Macro_F1_present_classes"],
            ascending=[True, False],
        )
    )

    return summary_df, per_class_df, confusion_matrices, fitted_model

## Section 13 — Centralized Logistic Regression baseline

This model is trained on the pooled training data from England, Ireland, and USA. It provides a simple centralized linear baseline.

The model uses `class_weight="balanced"` to reduce the influence of dominant classes in the pooled training set.

In [26]:
# 13.1 — Define and run centralized Logistic Regression

central_logistic_model = LogisticRegression(
    class_weight="balanced",
    max_iter=700,
    solver="lbfgs",
    random_state=SEED,
    n_jobs=-1,
)

central_logreg_summary_df, central_logreg_per_class_df, central_logreg_cms, central_logreg_model = run_centralized_model(
    model_name="Logistic_Regression",
    model=central_logistic_model,
)

_ = save_table(
    central_logreg_summary_df,
    "centralized_logistic_regression_summary_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_logreg_per_class_df,
    "centralized_logistic_regression_per_class_metrics",
    directory=METRIC_DIR,
)


CENTRALIZED BASELINE — Logistic_Regression

Training on pooled data: England + Ireland + USA
Pooled train shape: (134530, 96)
Finished centralized training for Logistic_Regression.

--------------------------------------------------------------------------------
Evaluating centralized Logistic_Regression on client: England
--------------------------------------------------------------------------------
✓ saved table: centralized_logistic_regression_england_val_confusion_matrix.csv
✓ saved table: centralized_logistic_regression_england_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized Logistic_Regression on client: Ireland
--------------------------------------------------------------------------------
✓ saved table: centralized_logistic_regression_ireland_val_confusion_matrix.csv
✓ saved table: centralized_logistic_regression_ireland_test_confusion_matrix.csv

---------------------------------------------

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,Logistic_Regression,Centralized,Pooled,England,val,8490,0.6161,0.5792,0.5998,0.5998,0.6092,"Excellent, Good, Fair, Marginal, Poor"
1,Logistic_Regression,Centralized,Pooled,England,test,11942,0.6055,0.4854,0.5212,0.5212,0.5932,"Excellent, Good, Fair, Marginal, Poor"
2,Logistic_Regression,Centralized,Pooled,Ireland,val,4296,0.4574,0.2330,0.1525,0.1906,0.5782,"Excellent, Good, Fair, Marginal"
3,Logistic_Regression,Centralized,Pooled,Ireland,test,5887,0.5210,0.3217,0.1576,0.2626,0.6354,"Excellent, Good, Fair"
4,Logistic_Regression,Centralized,Pooled,USA,val,1770,0.8746,0.3144,0.1896,0.3160,0.8789,"Excellent, Good, Fair"
5,Logistic_Regression,Centralized,Pooled,USA,test,1979,0.7792,0.4417,0.1838,0.4596,0.8091,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,Logistic_Regression,Centralized,Pooled,England,test,11942,0.6055,0.4854,0.5212,0.5212,0.5932,"Excellent, Good, Fair, Marginal, Poor"
3,Logistic_Regression,Centralized,Pooled,Ireland,test,5887,0.5210,0.3217,0.1576,0.2626,0.6354,"Excellent, Good, Fair"
5,Logistic_Regression,Centralized,Pooled,USA,test,1979,0.7792,0.4417,0.1838,0.4596,0.8091,"Excellent, Good"


✓ saved table: centralized_logistic_regression_summary_metrics.csv
✓ saved table: centralized_logistic_regression_per_class_metrics.csv


### Interpretation of the centralized Logistic Regression baseline

The centralized Logistic Regression baseline performs worse than the local Logistic Regression models for all three clients. Although the centralized model is trained on a larger pooled training set, it must learn one global linear decision boundary across England, Ireland, and USA.

This result suggests that pooling heterogeneous client data does not automatically improve performance when the model is too simple. The active clients have different class distributions and feature distributions, so a single linear model cannot easily adapt to all country-specific patterns.

The largest degradation is observed for USA, where the local Logistic Regression model was relatively strong but the centralized model performs much worse. This supports the interpretation that the clients are non-IID and that model capacity and client heterogeneity are important factors.

Overall, centralized Logistic Regression is a useful reference baseline, but it is not suitable as the main centralized classical ML model for this dataset. The next step is to evaluate centralized tree-based models, starting with Random Forest.

## Section 14 — Centralized Random Forest baseline

The centralized Random Forest model is trained on the pooled training data from England, Ireland, and USA.

Unlike Logistic Regression, Random Forest is a nonlinear model. Therefore, this experiment checks whether a more flexible centralized classical ML model can better handle the heterogeneous client data.

The model uses `class_weight="balanced_subsample"` to reduce the influence of dominant classes during tree construction.

In [27]:
# 14.1 — Define and run centralized Random Forest

central_random_forest_model = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_leaf=2,
    class_weight="balanced_subsample",
    random_state=SEED,
    n_jobs=-1,
)

central_rf_summary_df, central_rf_per_class_df, central_rf_cms, central_rf_model = run_centralized_model(
    model_name="Random_Forest",
    model=central_random_forest_model,
)

_ = save_table(
    central_rf_summary_df,
    "centralized_random_forest_summary_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_rf_per_class_df,
    "centralized_random_forest_per_class_metrics",
    directory=METRIC_DIR,
)


CENTRALIZED BASELINE — Random_Forest

Training on pooled data: England + Ireland + USA
Pooled train shape: (134530, 96)
Finished centralized training for Random_Forest.

--------------------------------------------------------------------------------
Evaluating centralized Random_Forest on client: England
--------------------------------------------------------------------------------
✓ saved table: centralized_random_forest_england_val_confusion_matrix.csv
✓ saved table: centralized_random_forest_england_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized Random_Forest on client: Ireland
--------------------------------------------------------------------------------
✓ saved table: centralized_random_forest_ireland_val_confusion_matrix.csv
✓ saved table: centralized_random_forest_ireland_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating c

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,Random_Forest,Centralized,Pooled,England,val,8490,0.8000,0.6960,0.7301,0.7301,0.7992,"Excellent, Good, Fair, Marginal, Poor"
1,Random_Forest,Centralized,Pooled,England,test,11942,0.8211,0.6296,0.6636,0.6636,0.8186,"Excellent, Good, Fair, Marginal, Poor"
2,Random_Forest,Centralized,Pooled,Ireland,val,4296,0.9013,0.3510,0.2970,0.3713,0.8889,"Excellent, Good, Fair, Marginal"
3,Random_Forest,Centralized,Pooled,Ireland,test,5887,0.9156,0.4670,0.2993,0.4988,0.9040,"Excellent, Good, Fair"
4,Random_Forest,Centralized,Pooled,USA,val,1770,0.9605,0.5566,0.3394,0.5657,0.9594,"Excellent, Good, Fair"
5,Random_Forest,Centralized,Pooled,USA,test,1979,0.9272,0.7302,0.3091,0.7727,0.9212,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,Random_Forest,Centralized,Pooled,England,test,11942,0.8211,0.6296,0.6636,0.6636,0.8186,"Excellent, Good, Fair, Marginal, Poor"
3,Random_Forest,Centralized,Pooled,Ireland,test,5887,0.9156,0.4670,0.2993,0.4988,0.9040,"Excellent, Good, Fair"
5,Random_Forest,Centralized,Pooled,USA,test,1979,0.9272,0.7302,0.3091,0.7727,0.9212,"Excellent, Good"


✓ saved table: centralized_random_forest_summary_metrics.csv
✓ saved table: centralized_random_forest_per_class_metrics.csv


### Interpretation of the centralized Random Forest baseline

The centralized Random Forest model performs much better than centralized Logistic Regression, showing that nonlinear tree-based models are more suitable for the pooled heterogeneous dataset than a simple linear classifier.

However, centralized Random Forest does not outperform the local Random Forest models. The test macro-F1 over present classes decreases slightly for England, Ireland, and USA compared with the local setting. This suggests that although pooling increases the amount of training data, it can also reduce client-specific specialization.

The largest decrease appears for England, which is the only client with all five classes in the test set. This indicates that the pooled model may lose part of England’s local class structure when trained together with Ireland and USA, which are more dominated by Excellent and Good classes.

Overall, centralized Random Forest is a strong centralized classical ML baseline, but the local Random Forest remains slightly better. The next step is to evaluate centralized XGBoost to check whether boosted trees can benefit more from pooled training data.

## Section 15 — Centralized XGBoost baseline

The centralized XGBoost model is trained on the pooled training data from England, Ireland, and USA.

XGBoost is a boosted tree model, meaning that trees are built sequentially and each new tree tries to correct the errors of the previous trees. This makes it a strong classical machine learning model for tabular data.

Since our input sequences were flattened into 96 tabular features, XGBoost is suitable for this benchmark. Balanced sample weights are passed during training to reduce the effect of class imbalance in the pooled dataset.

In [28]:
# 15.1 — Define and run centralized XGBoost

central_xgboost_model = XGBClassifier(
    n_estimators=300,
    max_depth=4,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    objective="multi:softprob",
    eval_metric="mlogloss",
    tree_method="hist",
    random_state=SEED,
    n_jobs=-1,
)

central_xgb_summary_df, central_xgb_per_class_df, central_xgb_cms, central_xgb_model = run_centralized_model(
    model_name="XGBoost",
    model=central_xgboost_model,
)

_ = save_table(
    central_xgb_summary_df,
    "centralized_xgboost_summary_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_xgb_per_class_df,
    "centralized_xgboost_per_class_metrics",
    directory=METRIC_DIR,
)


CENTRALIZED BASELINE — XGBoost

Training on pooled data: England + Ireland + USA
Pooled train shape: (134530, 96)
Finished centralized training for XGBoost.

--------------------------------------------------------------------------------
Evaluating centralized XGBoost on client: England
--------------------------------------------------------------------------------
✓ saved table: centralized_xgboost_england_val_confusion_matrix.csv
✓ saved table: centralized_xgboost_england_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized XGBoost on client: Ireland
--------------------------------------------------------------------------------
✓ saved table: centralized_xgboost_ireland_val_confusion_matrix.csv
✓ saved table: centralized_xgboost_ireland_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized XGBoost on client: USA
--------------

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Centralized,Pooled,England,val,8490,0.7775,0.7292,0.7308,0.7308,0.7819,"Excellent, Good, Fair, Marginal, Poor"
1,XGBoost,Centralized,Pooled,England,test,11942,0.8017,0.6595,0.6699,0.6699,0.8050,"Excellent, Good, Fair, Marginal, Poor"
2,XGBoost,Centralized,Pooled,Ireland,val,4296,0.9015,0.3819,0.3121,0.3901,0.8992,"Excellent, Good, Fair, Marginal"
3,XGBoost,Centralized,Pooled,Ireland,test,5887,0.9123,0.5380,0.3229,0.5381,0.9123,"Excellent, Good, Fair"
4,XGBoost,Centralized,Pooled,USA,val,1770,0.9621,0.5932,0.3474,0.5789,0.9629,"Excellent, Good, Fair"
5,XGBoost,Centralized,Pooled,USA,test,1979,0.9207,0.7456,0.3094,0.7735,0.9190,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,XGBoost,Centralized,Pooled,England,test,11942,0.8017,0.6595,0.6699,0.6699,0.8050,"Excellent, Good, Fair, Marginal, Poor"
3,XGBoost,Centralized,Pooled,Ireland,test,5887,0.9123,0.5380,0.3229,0.5381,0.9123,"Excellent, Good, Fair"
5,XGBoost,Centralized,Pooled,USA,test,1979,0.9207,0.7456,0.3094,0.7735,0.9190,"Excellent, Good"


✓ saved table: centralized_xgboost_summary_metrics.csv
✓ saved table: centralized_xgboost_per_class_metrics.csv


### Interpretation of the centralized XGBoost baseline

The centralized XGBoost model is the strongest centralized classical ML model so far. Compared with centralized Random Forest, it improves macro-F1 over present classes for England and Ireland and gives a very similar result for USA.

However, centralized XGBoost does not improve all clients compared with the local XGBoost setting. It improves Ireland, suggesting that this client benefits from pooled training information. In contrast, England and USA perform better with local XGBoost, indicating that client-specific specialization remains important.

This result shows that centralized pooling is not uniformly beneficial. It can help some clients, especially those with fewer or more imbalanced training samples, but it may reduce performance for clients with strong local structure.

## Section 16 — Centralized CatBoost baseline

The centralized CatBoost model is trained on the pooled training data from all active clients and then evaluated separately on each client. This provides a second boosted-tree reference in the centralized setting, alongside centralized XGBoost.


In [29]:
# 16.1 — Define and run centralized CatBoost

central_catboost_model = CatBoostClassifier(
    iterations=300,
    depth=6,
    learning_rate=0.05,
    loss_function="MultiClass",
    auto_class_weights="Balanced",
    random_seed=SEED,
    verbose=False,
    allow_writing_files=False,
    thread_count=-1,
)

central_cat_summary_df, central_cat_per_class_df, central_cat_cms, central_cat_model = run_centralized_model(
    model_name="CatBoost",
    model=central_catboost_model,
)

_ = save_table(
    central_cat_summary_df,
    "centralized_catboost_summary_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_cat_per_class_df,
    "centralized_catboost_per_class_metrics",
    directory=METRIC_DIR,
)


CENTRALIZED BASELINE — CatBoost

Training on pooled data: England + Ireland + USA
Pooled train shape: (134530, 96)
Finished centralized training for CatBoost.

--------------------------------------------------------------------------------
Evaluating centralized CatBoost on client: England
--------------------------------------------------------------------------------
✓ saved table: centralized_catboost_england_val_confusion_matrix.csv
✓ saved table: centralized_catboost_england_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized CatBoost on client: Ireland
--------------------------------------------------------------------------------
✓ saved table: centralized_catboost_ireland_val_confusion_matrix.csv
✓ saved table: centralized_catboost_ireland_test_confusion_matrix.csv

--------------------------------------------------------------------------------
Evaluating centralized CatBoost on client: USA
-----

,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,CatBoost,Centralized,Pooled,England,val,8490,0.7694,0.7192,0.7209,0.7209,0.7741,"Excellent, Good, Fair, Marginal, Poor"
1,CatBoost,Centralized,Pooled,England,test,11942,0.8005,0.6343,0.6561,0.6561,0.8034,"Excellent, Good, Fair, Marginal, Poor"
2,CatBoost,Centralized,Pooled,Ireland,val,4296,0.9001,0.3609,0.3033,0.3792,0.8949,"Excellent, Good, Fair, Marginal"
3,CatBoost,Centralized,Pooled,Ireland,test,5887,0.9059,0.4760,0.3004,0.5006,0.9028,"Excellent, Good, Fair"
4,CatBoost,Centralized,Pooled,USA,val,1770,0.9588,0.5848,0.3427,0.5712,0.9601,"Excellent, Good, Fair"
5,CatBoost,Centralized,Pooled,USA,test,1979,0.9045,0.7281,0.2973,0.7433,0.9073,"Excellent, Good"



Test metrics only:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
1,CatBoost,Centralized,Pooled,England,test,11942,0.8005,0.6343,0.6561,0.6561,0.8034,"Excellent, Good, Fair, Marginal, Poor"
3,CatBoost,Centralized,Pooled,Ireland,test,5887,0.9059,0.4760,0.3004,0.5006,0.9028,"Excellent, Good, Fair"
5,CatBoost,Centralized,Pooled,USA,test,1979,0.9045,0.7281,0.2973,0.7433,0.9073,"Excellent, Good"


✓ saved table: centralized_catboost_summary_metrics.csv
✓ saved table: centralized_catboost_per_class_metrics.csv


### Interpretation of the centralized CatBoost baseline

The centralized CatBoost model performs better than centralized Logistic Regression, confirming again that nonlinear tree-based models are more suitable for the pooled data than a linear classifier.

However, centralized CatBoost does not outperform centralized XGBoost. XGBoost gives higher macro-F1 over present classes for England, Ireland, and USA. Compared with local CatBoost, the centralized version slightly improves Ireland but decreases performance for England and USA.

This result supports the same conclusion observed with centralized XGBoost: pooling data can help some clients, especially Ireland, but it may reduce performance for clients where local patterns are stronger.

## Section 17 — Centralized classical ML comparison

This section combines the centralized results from Logistic Regression, Random Forest, XGBoost, and CatBoost.

The main comparison metric is macro-F1 over present classes. This metric is used because some client test sets do not contain all five classes, especially Ireland and USA.

In [30]:
# 17.1 — Combine centralized classical ML results

central_classical_summary_df = pd.concat(
    [
        central_logreg_summary_df,
        central_rf_summary_df,
        central_xgb_summary_df,
        central_cat_summary_df,
    ],
    ignore_index=True,
)

central_classical_per_class_df = pd.concat(
    [
        central_logreg_per_class_df,
        central_rf_per_class_df,
        central_xgb_per_class_df,
        central_cat_per_class_df,
    ],
    ignore_index=True,
)

central_classical_test_df = central_classical_summary_df[
    central_classical_summary_df["Split"] == "test"
].copy()

central_classical_test_df = central_classical_test_df.sort_values(
    by=["Eval_client", "Macro_F1_present_classes"],
    ascending=[True, False],
).reset_index(drop=True)

print("Centralized classical ML test results ranked by client and Macro-F1 present:")
display(central_classical_test_df)

_ = save_table(
    central_classical_summary_df,
    "centralized_classical_ml_all_summary_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_classical_per_class_df,
    "centralized_classical_ml_all_per_class_metrics",
    directory=METRIC_DIR,
)

_ = save_table(
    central_classical_test_df,
    "centralized_classical_ml_test_results_ranked",
    directory=METRIC_DIR,
)

Centralized classical ML test results ranked by client and Macro-F1 present:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Centralized,Pooled,England,test,11942,0.8017,0.6595,0.6699,0.6699,0.8050,"Excellent, Good, Fair, Marginal, Poor"
1,Random_Forest,Centralized,Pooled,England,test,11942,0.8211,0.6296,0.6636,0.6636,0.8186,"Excellent, Good, Fair, Marginal, Poor"
2,CatBoost,Centralized,Pooled,England,test,11942,0.8005,0.6343,0.6561,0.6561,0.8034,"Excellent, Good, Fair, Marginal, Poor"
3,Logistic_Regression,Centralized,Pooled,England,test,11942,0.6055,0.4854,0.5212,0.5212,0.5932,"Excellent, Good, Fair, Marginal, Poor"
4,XGBoost,Centralized,Pooled,Ireland,test,5887,0.9123,0.5380,0.3229,0.5381,0.9123,"Excellent, Good, Fair"
5,CatBoost,Centralized,Pooled,Ireland,test,5887,0.9059,0.4760,0.3004,0.5006,0.9028,"Excellent, Good, Fair"
6,Random_Forest,Centralized,Pooled,Ireland,test,5887,0.9156,0.4670,0.2993,0.4988,0.9040,"Excellent, Good, Fair"
7,Logistic_Regression,Centralized,Pooled,Ireland,test,5887,0.5210,0.3217,0.1576,0.2626,0.6354,"Excellent, Good, Fair"
8,XGBoost,Centralized,Pooled,USA,test,1979,0.9207,0.7456,0.3094,0.7735,0.9190,"Excellent, Good"
9,Random_Forest,Centralized,Pooled,USA,test,1979,0.9272,0.7302,0.3091,0.7727,0.9212,"Excellent, Good"


✓ saved table: centralized_classical_ml_all_summary_metrics.csv
✓ saved table: centralized_classical_ml_all_per_class_metrics.csv
✓ saved table: centralized_classical_ml_test_results_ranked.csv


In [31]:
# 17.2 — Best centralized classical ML model per client

best_central_classical_df = (
    central_classical_test_df
    .sort_values(["Eval_client", "Macro_F1_present_classes"], ascending=[True, False])
    .groupby("Eval_client", as_index=False)
    .head(1)
    .reset_index(drop=True)
)

print("Best centralized classical ML model per client:")
display(best_central_classical_df)

_ = save_table(
    best_central_classical_df,
    "best_centralized_classical_ml_model_per_client",
    directory=METRIC_DIR,
)

Best centralized classical ML model per client:


,Model,Strategy,Train_client,Eval_client,Split,N,Accuracy,Balanced_accuracy,Macro_F1_all_classes,Macro_F1_present_classes,Weighted_F1,Present_classes
0,XGBoost,Centralized,Pooled,England,test,11942,0.8017,0.6595,0.6699,0.6699,0.8050,"Excellent, Good, Fair, Marginal, Poor"
1,XGBoost,Centralized,Pooled,Ireland,test,5887,0.9123,0.5380,0.3229,0.5381,0.9123,"Excellent, Good, Fair"
2,XGBoost,Centralized,Pooled,USA,test,1979,0.9207,0.7456,0.3094,0.7735,0.9190,"Excellent, Good"


✓ saved table: best_centralized_classical_ml_model_per_client.csv


In [32]:
# 17.3 — Compact comparison table: centralized Macro-F1 present classes

central_macro_f1_pivot_df = central_classical_test_df.pivot_table(
    index="Eval_client",
    columns="Model",
    values="Macro_F1_present_classes",
    aggfunc="first",
)

central_macro_f1_pivot_df = central_macro_f1_pivot_df.reindex(ACTIVE_CLIENTS)

print("Centralized classical ML comparison using Macro-F1 over present classes:")
display(central_macro_f1_pivot_df.round(4))

_ = save_table(
    central_macro_f1_pivot_df.reset_index(),
    "centralized_classical_ml_macro_f1_present_pivot",
    directory=METRIC_DIR,
)

Centralized classical ML comparison using Macro-F1 over present classes:


Model,CatBoost,Logistic_Regression,Random_Forest,XGBoost
Eval_client,,,,
England,0.6561,0.5212,0.6636,0.6699
Ireland,0.5006,0.2626,0.4988,0.5381
USA,0.7433,0.4596,0.7727,0.7735


✓ saved table: centralized_classical_ml_macro_f1_present_pivot.csv


### Interpretation of centralized classical ML baselines

The centralized classical ML results show that XGBoost is the strongest model in the pooled-data setting. It obtains the highest macro-F1 over present classes for England, Ireland, and USA. This confirms that boosted tree models are more suitable than a simple linear classifier for the heterogeneous water-quality data.

However, centralized training does not improve all clients equally. Compared with the best local classical ML models, centralized XGBoost improves Ireland but decreases performance for England and USA. This suggests that pooling data can help clients with smaller or more imbalanced local datasets, but it may reduce client-specific specialization for stronger clients.

Overall, the centralized classical ML benchmark provides an important reference point. It shows that more training data is not automatically better; model type and client heterogeneity strongly affect the final performance.

## Section 18 — Local vs centralized classical ML comparison

This section compares the best local classical ML model with the best centralized classical ML model for each client.

This comparison is important because it shows whether pooling data improves classical ML performance or whether client-specific local training remains better.

In [33]:
# 18.1 — Best local classical ML vs best centralized classical ML

local_best_classical_compare_df = best_local_classical_df[
    ["Eval_client", "Model", "Macro_F1_present_classes"]
].copy()

local_best_classical_compare_df = local_best_classical_compare_df.rename(
    columns={
        "Model": "Best_local_classical_model",
        "Macro_F1_present_classes": "Best_local_macro_F1_present",
    }
)

central_best_classical_compare_df = best_central_classical_df[
    ["Eval_client", "Model", "Macro_F1_present_classes"]
].copy()

central_best_classical_compare_df = central_best_classical_compare_df.rename(
    columns={
        "Model": "Best_centralized_classical_model",
        "Macro_F1_present_classes": "Best_centralized_macro_F1_present",
    }
)

local_vs_central_classical_df = local_best_classical_compare_df.merge(
    central_best_classical_compare_df,
    on="Eval_client",
    how="inner",
)

local_vs_central_classical_df["Centralized_minus_local"] = (
    local_vs_central_classical_df["Best_centralized_macro_F1_present"]
    - local_vs_central_classical_df["Best_local_macro_F1_present"]
).round(4)

local_vs_central_classical_df["Better_strategy"] = np.where(
    local_vs_central_classical_df["Centralized_minus_local"] > 0,
    "Centralized",
    "Local",
)

print("Best local classical ML vs best centralized classical ML:")
display(local_vs_central_classical_df)

_ = save_table(
    local_vs_central_classical_df,
    "best_local_vs_best_centralized_classical_ml",
    directory=METRIC_DIR,
)

Best local classical ML vs best centralized classical ML:


,Eval_client,Best_local_classical_model,Best_local_macro_F1_present,Best_centralized_classical_model,Best_centralized_macro_F1_present,Centralized_minus_local,Better_strategy
0,England,XGBoost,0.7145,XGBoost,0.6699,-0.0446,Local
1,Ireland,Random_Forest,0.5030,XGBoost,0.5381,0.0351,Centralized
2,USA,XGBoost,0.7889,XGBoost,0.7735,-0.0154,Local


✓ saved table: best_local_vs_best_centralized_classical_ml.csv


## Final note — handoff to Notebook 09

Notebook 08 stops at the classical ML benchmark. It does not perform the final global ranking across all model families. The saved tables and prediction files from this notebook will be used later in Notebook 09, where classical ML, GRU, LSTM, Transformer, and federated results are compared in one unified framework.

This separation keeps the pipeline clean: Notebook 08 answers the classical ML question, while Notebook 09 answers the global comparison question.


In [34]:
# Save N08 outputs as a ZIP file

from pathlib import Path
import shutil
import os

# Main N08 output folder
OUTPUT_DIR = Path("/kaggle/working/classical_ml_baselines")

assert OUTPUT_DIR.exists(), f"Output folder not found: {OUTPUT_DIR}"

ZIP_BASE = Path("/kaggle/working/N08_classical_ml_baselines_outputs")
ZIP_PATH = Path(str(ZIP_BASE) + ".zip")

# Remove old zip if it exists
if ZIP_PATH.exists():
    ZIP_PATH.unlink()

# Create zip
shutil.make_archive(
    base_name=str(ZIP_BASE),
    format="zip",
    root_dir=str(OUTPUT_DIR)
)

print("✅ ZIP file created successfully:")
print(ZIP_PATH)
print(f"Size: {ZIP_PATH.stat().st_size / (1024*1024):.2f} MB")

# Show included files
print("\nIncluded files:")
for root, dirs, files in os.walk(OUTPUT_DIR):
    for file in files:
        file_path = Path(root) / file
        print(" -", file_path.relative_to(OUTPUT_DIR))

✅ ZIP file created successfully:
/kaggle/working/N08_classical_ml_baselines_outputs.zip
Size: 389.36 MB

Included files:
 - audit/02_tabular_shape_verification.csv
 - audit/03_class_distribution_audit.csv
 - audit/centralized_training_class_distribution.csv
 - audit/01_tensor_shape_verification.csv
 - models/local_xgboost_england.joblib
 - models/centralized_random_forest.joblib
 - models/local_logistic_regression_usa.joblib
 - models/local_catboost_usa.joblib
 - models/local_random_forest_usa.joblib
 - models/local_logistic_regression_ireland.joblib
 - models/local_logistic_regression_england.joblib
 - models/centralized_logistic_regression.joblib
 - models/local_xgboost_usa.joblib
 - models/local_catboost_ireland.joblib
 - models/centralized_catboost.joblib
 - models/centralized_xgboost.joblib
 - models/local_catboost_england.joblib
 - models/local_random_forest_england.joblib
 - models/local_xgboost_ireland.joblib
 - models/local_random_forest_ireland.joblib
 - predictions/centraliz